In [1]:
import json
def evaluate(file_path,ground_truth="Non-binary"):
    with open(file_path, 'r') as f:
        qa_results = json.load(f)
    
    # Calculate statistics
    total = len(qa_results)
    yes = sum(1 for qa in qa_results if qa['verified_answer']=="Yes")
    no = sum(1 for qa in qa_results if qa['verified_answer']=="No")
    retrieval_collapse = sum(1 for qa in qa_results if qa['verified_answer']=='Not enough information')
    
    stats = {"Yes": yes, "No": no, "Not enough information": retrieval_collapse}
    
    # Calculate RAG performance metrics
    retrieval_success = total - retrieval_collapse # = yes + no

    rag_scores = {
        # the retrieved context is either none or insufficient to answer the question, leading to "Not enough information" answer
        "failure_rate": retrieval_collapse / total if total > 0 else 0, 
        # there is sufficient retrieved context leading to a "Yes" or "No" answer, but the final answer may still be incorrect due to reasoning errors from zero-shot classification
        "success_rate": retrieval_success / total if total > 0 else 0  
    }

    if ground_truth == "Yes": 
        # true positive is "Yes", false negative is "No" or "Not enough information", and no false positive
        # recall = accuracy = tp / (tp + fn) = true positve rate
        false_negative_cases = no + retrieval_collapse
        recall = yes / (yes + false_negative_cases) if (false_negative_cases) > 0 else 0
        precision = 1 # precision = TP / (TP + FP), and FP = 0 in this case
        
        accuracy = yes / total if total > 0 else 0
        error = false_negative_cases / total if total > 0 else 0

        f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0 
        metrics = {"precision": precision, "recall": recall,"accuracy": accuracy, "error": error, "f1_score": f1_score}
    
    elif ground_truth == "No": 
        # true positive is "No", false negative is "Yes" or "Not enough information", and no false positive
        # recall = accuracy = tp / (tp + fn) = true positve rate
        false_negative_cases = yes + retrieval_collapse
        recall = no / (no + false_negative_cases) if (no + false_negative_cases) > 0 else 0
        precision = 1 # precision = TP / (TP + FP), and FP = 0 in this case

        accuracy = no / total if total > 0 else 0
        error = false_negative_cases / total if total > 0 else 0
        f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0 

        metrics = {"precision": precision, "recall": recall,"accuracy": accuracy, "error": error, "f1_score": f1_score}
    
    elif ground_truth == "Non-binary":
        # Case of PubMedQA_labeled where the ground truth is non-binary
        # true positive = qa['final_answer'] matches ground truth final_decision 
        # false negative = "Not enough information" when the ground truth is "yes" or "no"
        # and false positive = "not enough information" when the ground truth is "maybe"
        with open('../pqa_labeled_mapping.json', 'r') as f:
            ground_truth_mapping = json.load(f)
        
        # mapping questions to their corresponding ground truth final_decision
        question_to_decision = {item['question']: item['final_decision'] for item in ground_truth_mapping}

        RED = "\033[31m"
        RESET = "\033[0m"
        
        fn = 0
        fp = 0
        tn = 0
        tp_yes = 0
        tp_no = 0
        for qa in qa_results:
            # positive = yes/no; negative = not enough information
            # tp = actually is & predicted as yes/no 
            # tn = actually is 'maybe' & predicted as 'not enough information'
            # fp = actually is 'maybe' but predicted as 'yes'/'no'
            # fn = actually is 'yes'/'no' but predicted as 'not enough information'
            final_answer = qa['verified_answer'].lower()
            true_label = question_to_decision.get(qa['question'], "Unknown")
            if final_answer == true_label and true_label == "yes":
                tp_yes += 1
            if true_label != "maybe" and final_answer == "not enough information":
                fn += 1
                # print(f"Real Label: {RED}{true_label}{RESET}")
                # print(f"Predicted Label: {RED}{final_answer}{RESET}")
            elif true_label == "maybe" and final_answer != "not enough information":
                fp += 1
                # print(f"Real Label: {RED}{true_label}{RESET}")
                # print(f"Predicted Label: {RED}{final_answer}{RESET}")
            elif true_label == "maybe" and final_answer == "not enough information":
                tn += 1
                # print(f"Real Label: \033[32m{true_label}{RESET}")
                # print(f"Predicted Label: \033[32m{final_answer}{RESET}")
            else:
                tp_no += 1
                # print(f"Real Label: \033[32m{true_label}{RESET}")
                # print(f"Predicted Label: \033[32m{final_answer}{RESET}")

        

        unlabeled = len(question_to_decision) - tp_yes - tp_no - fp - tn - fn
        print(f"Unclassified: {unlabeled}")
        precision = {"Yes/No":tp_yes + tp_no/(tp_yes + tp_no+fp), "Not enough information": tn/(tn+fn)}
        recall = {"Yes/No":tp_yes + tp_no/(tp_yes + tp_no + fn), "Not enough information": tn/(tn + fp)}
        
        macro_true_positives = (tp_yes + tp_no + tn) # correctly labeled as is
        misclassified = (fp + fn) # = macro_false_negatives (i.e misclassification between "Not enough information" and "Yes/No")
        
        # statistics results / confusion matrix
        stats = {"TP": tp_yes + tp_no, "TN": tn, "FP": fp, "FN": fn}
        print(f"True yes: {tp_yes}, True no: {tp_no}")
        # precision = recall
        macro_precision = macro_true_positives / (macro_true_positives + fp) if (macro_true_positives + fp) > 0 else 0
        macro_recall = macro_true_positives / (macro_true_positives + fn) if (macro_true_positives + fn) > 0 else 0
        accuracy = macro_true_positives / (macro_true_positives + misclassified) if (macro_true_positives + misclassified) > 0 else 0
        error = misclassified / (macro_true_positives + misclassified) if (macro_true_positives + misclassified) > 0 else 0

        
        f1_score = 2 * (macro_precision * macro_recall) / (macro_precision + macro_recall) if (macro_precision + macro_recall) > 0 else 0 

        metrics = {"macro_precision": macro_precision, "macro_recall": macro_recall,"accuracy": accuracy, "error": error, "f1_score": f1_score}
    

    return stats, rag_scores, metrics

In [2]:
def confusion_matrix(stats,ground_truth="Non-binary"):
    print("Statistics Results:")
    print(f"Total Questions: {sum(stats.values())}")
    if ground_truth == "Yes":
        tp = stats["Yes"]
        fn = stats["No"] + stats["Not enough information"]
        fp = 0
        tn = 0
        print(f"Yes: {tp}, No: {stats['No']}, Not enough information: {stats['Not enough information']}")
    elif ground_truth == "No":
        tp = stats["No"]
        fn = stats["Yes"] + stats["Not enough information"]
        fp = 0
        tn = 0
        print(f"Yes: {stats["Yes"]}, No: {tp}, Not enough information: {stats['Not enough information']}")
    elif ground_truth == "Non-binary":
        tp = stats["TP"]
        fn = stats["FN"]
        fp = stats["FP"]
        tn = stats["TN"]
        print(f"Yes/No: {tp}, Not enough information: {tn}, Misclassified: {fp} as 'yes'/'no' + {fn} as 'not enough information'")

    return {"TP": tp, "FN": fn, "FP": fp, "TN": tn}

def print_confusion_matrix(conf_matrix):
    print("Confusion Matrix:")
    print("\t Predicted")
    print("\t   P | N")
    print("-"*20)
    print(f"Actual P  {conf_matrix['TP']} | {conf_matrix['FN']}  ")
    print(f"Actual N  {conf_matrix['FP']} |  {conf_matrix['TN']}  ")
    print("-"*20)
    

In [3]:
def print_evaluation(rag_scores, metrics,ground_truth="Non-binary"):
    print("Evaluation Results:")
    
    print("\nRAG Performance Metrics:")
    print(f"Failure Rate (Not enough information): {rag_scores['failure_rate']:.2%}")
    print(f"Success Rate (Yes/No): {rag_scores['success_rate']:.2%}")
    
    print("\nPrecision and Recall:")
    if ground_truth != "Non-binary":
        print(f"Precision: {metrics['precision']:.2%}")
        print(f"Recall: {metrics['recall']:.2%}")
    else:
        print(f"Precision: {metrics['macro_precision']:.2%}")
        print(f"Recall: {metrics['macro_recall']:.2%}")
    print(f"F1 Score: {metrics['f1_score']:.2%}")
    print(f"Accuracy: {metrics['accuracy']:.2%}")
    print(f"Error: {metrics['error']:.2%}")

    
    

In [5]:
file_path = "noQA_results.json"
#file_path = "noQA_3R.json"
no_stats, no_rag_scores, no_metrics = evaluate(file_path, ground_truth="No")
print_confusion_matrix(confusion_matrix(no_stats, ground_truth="No"))
print_evaluation(no_rag_scores, no_metrics,"No")

Statistics Results:
Total Questions: 93
Yes: 4, No: 23, Not enough information: 66
Confusion Matrix:
	 Predicted
	   P | N
--------------------
Actual P  23 | 70  
Actual N  0 |  0  
--------------------
Evaluation Results:

RAG Performance Metrics:
Failure Rate (Not enough information): 70.97%
Success Rate (Yes/No): 29.03%

Precision and Recall:
Precision: 100.00%
Recall: 24.73%
F1 Score: 39.66%
Accuracy: 24.73%
Error: 75.27%


In [6]:
file_path = "yesQA_results.json"
#file_path = "yesQA_3R.json"
yes_stats, yes_rag_scores, yes_metrics = evaluate(file_path, ground_truth="Yes")
print_confusion_matrix(confusion_matrix(yes_stats, ground_truth="Yes"))
print_evaluation(yes_rag_scores, yes_metrics,"Yes")

Statistics Results:
Total Questions: 93
Yes: 8, No: 6, Not enough information: 79
Confusion Matrix:
	 Predicted
	   P | N
--------------------
Actual P  8 | 85  
Actual N  0 |  0  
--------------------
Evaluation Results:

RAG Performance Metrics:
Failure Rate (Not enough information): 84.95%
Success Rate (Yes/No): 15.05%

Precision and Recall:
Precision: 100.00%
Recall: 8.60%
F1 Score: 15.84%
Accuracy: 8.60%
Error: 91.40%


In [126]:
file_path = "pqa_labeled_3R.json"
stats, rag_scores, metrics = evaluate(file_path, ground_truth="Non-binary")
print_confusion_matrix(confusion_matrix(stats, ground_truth="Non-binary"))
print_evaluation(rag_scores, metrics,"Non-binary")

Unclassified: 0
True yes: 0, True no: 25
Statistics Results:
Total Questions: 1000
Yes/No: 25, Not enough information: 108, Misclassified: 2 as 'yes'/'no' + 865 as 'not enough information'
Confusion Matrix:
	 Predicted
	   P | N
--------------------
Actual P  25 | 865  
Actual N  2 |  108  
--------------------
Evaluation Results:

RAG Performance Metrics:
Failure Rate (Not enough information): 97.30%
Success Rate (Yes/No): 2.70%

Precision and Recall:
Precision: 98.52%
Recall: 13.33%
F1 Score: 23.48%
Accuracy: 13.30%
Error: 86.70%


### IMPORT PUBMEDQA BENCHMARK FOR EVALUATION

In [ ]:
import pandas as pd

df = pd.read_parquet("hf://datasets/qiaojin/PubMedQA/pqa_labeled/train-00000-of-00001.parquet")
df.columns
print(df['final_decision'].value_counts())

pubmedQA_set = df.to_dict(orient='records')

In [ ]:
import numpy as np
def convert(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    raise TypeError(f"Object of type {type(obj)} is not JSON serializable")

with open("../pqa_labeled.json",'w') as f:
    json.dump(pubmedQA_set, f,default=convert, indent=4)

In [ ]:
with open('../pqa_labeled.json', 'r') as f:
    pubmedQA_set = json.load(f)

# Create a mapping from final_answer to final_decision in the PubMedQA dataset
ground_truth_table = []
for qa in pubmedQA_set:
    ground_truth_table.append({
        "question": qa['question'],
        "final_decision": qa['final_decision']
    })
with open("../pqa_labeled_mapping.json",'w') as f:
    json.dump(ground_truth_table, f,default=convert, indent=4)

In [3]:
import json
with open('../pqa_labeled_mapping.json', 'r') as f:
    ground_truth_mapping = json.load(f)
with open('pqa_labeled_3R.json', 'r') as f:
    qa_results = json.load(f)
question_to_decision = {item['question']: item['final_decision'] for item in ground_truth_mapping}

print(len(question_to_decision))
print(len(qa_results))

RED = "\033[31m"
RESET = "\033[0m"
# matches = 0
# mismatches = 0
fn = 0
fp = 0
tn = 0
tp_yes = 0
tp_no = 0
for qa in qa_results:
    # positive = yes/no; negative = not enough information
    # tp = actually is & predicted as yes/no 
    # tn = actually is 'maybe' & predicted as 'not enough information'
    # fp = actually is 'maybe' but predicted as 'yes'/'no'
    # fn = actually is 'yes'/'no' but predicted as 'not enough information'
    final_answer = qa['final_answer'].lower()
    true_label = question_to_decision.get(qa['question'], "Unknown")
    
    if final_answer == true_label and true_label == "yes":
        tp_yes += 1
    if true_label != "maybe" and final_answer == "not enough information":
        fn += 1
        print(f"Real Label: {RED}{true_label}{RESET}")
        print(f"Predicted Label: {RED}{final_answer}{RESET}")
    elif true_label == "maybe" and final_answer != "not enough information":
        fp += 1
        print(f"Real Label: {RED}{true_label}{RESET}")
        print(f"Predicted Label: {RED}{final_answer}{RESET}")
    elif true_label == "maybe" and final_answer == "not enough information":
        tn += 1
        print(f"Real Label: \033[32m{true_label}{RESET}")
        print(f"Predicted Label: \033[32m{final_answer}{RESET}")
    else:
        tp_no += 1
        print(f"Real Label: \033[32m{true_label}{RESET}")
        print(f"Predicted Label: \033[32m{final_answer}{RESET}")
  

1000
1000
Real Label: yes
Predicted Label: not enough information
Real Label: no
Predicted Label: no
Real Label: yes
Predicted Label: not enough information
Real Label: no
Predicted Label: not enough information
Real Label: yes
Predicted Label: not enough information
Real Label: yes
Predicted Label: not enough information
Real Label: maybe
Predicted Label: not enough information
Real Label: no
Predicted Label: not enough information
Real Label: no
Predicted Label: not enough information
Real Label: yes
Predicted Label: not enough information
Real Label: yes
Predicted Label: not enough information
Real Label: no
Predicted Label: not enough information
Real Label: yes
Predicted Label: not enough information
Real Label: no
Predicted Label: not enough information
Real Label: yes
Predicted Label: not enough information
Real Label: yes
Predicted Label: no
Real Label: yes
Predicted Label: not enough information
Real Label: yes
Predicted Label: not enough information
Real Label: yes
Predicted 

In [5]:
print(f"FN cases: {fn}")
print(f"FP cases: {fp}")

print(f"TN cases: {tn}")
print(f"TP cases: {tp_yes + tp_no}\n\t 'Yes': {tp_yes};'No': {tp_no}")


total = fn + fp + tn + tp_yes + tp_no
print(total)

FN cases: 865
FP cases: 2
TN cases: 108
TP cases: 25
	 'Yes': 0;'No': 25
1000
